In [1]:
import numpy as np
from pathlib import Path


def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


X, y, patient = load_dataset()

In [ ]:
"""
AsymSENet: Asymmetric Squeeze-Excitation Network for EEG Seizure Detection
Implementation based on: "Enhancing Seizure Detection via Asymmetric Spatiotemporal
Feature Extraction" (Marihuan & Saavedra).

Pipeline (Modified to remove TCN):
    Raw EEG -> N segments -> Spatial Module (Asymmetric Conv + SE) x2
             -> Global Average Pooling
             -> Flatten all segments -> Classification Head (FC) -> Seizure / Normal
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------- #
# 1. Squeeze-and-Excitation block (2D, operates on conv feature maps)
# --------------------------------------------------------------------------- #
class SqueezeExcitation2d(nn.Module):
    """Channel-wise recalibration: squeeze (global avg pool) -> excitation
    (bottleneck MLP) -> scale (channel-wise multiplication). See Hu et al.,
    2019, as used between the convolutional layers of the Spatial Module."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        e = self.excite(s).view(b, c, 1, 1)
        return x * e


# --------------------------------------------------------------------------- #
# 2. Asymmetric convolutional block: Conv -> AvgPool -> SE -> Add & LayerNorm
# --------------------------------------------------------------------------- #
class AsymmetricConvBlock(nn.Module):
    """One stage of the Spatial Module (Section III-B).

    Applies an asymmetric (M x N) dilated convolution -> average pooling
    along the temporal axis -> Squeeze-and-Excitation recalibration -> a
    residual (skip) connection back to the block's input -> LayerNorm.

    A kernel height M equal to the current electrode/height dimension
    collapses that spatial axis to 1 in a single step (as in Spatial
    Module 1, kernel 22x4); a kernel height of 1 (Spatial Module 2,
    kernel 1x4) then only refines the temporal axis.
    """

    def __init__(self, in_channels, out_channels, kernel_size, dilation,
                 pool_size=(1, 2), se_reduction=16):
        super().__init__()
        kh, kw = kernel_size
        dh, dw = dilation

        # 'same'-style padding along the time axis; no padding along height
        # so that kh == current_height forces the electrode-dimension collapse
        # described in the paper (Eq. 1).
        pad_h = 0
        pad_w = ((kw - 1) * dw) // 2

        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=(kh, kw),
            dilation=(dh, dw), padding=(pad_h, pad_w),
        )
        self.pool = nn.AvgPool2d(kernel_size=pool_size, ceil_mode=True)
        self.se = SqueezeExcitation2d(out_channels, reduction=se_reduction)

        # Projection for the skip connection: mirrors the main branch's
        # height-collapsing behaviour (kernel/stride = kh along height) so the
        # residual add lines up in channels, height, and (after pooling) width.
        self.skip_proj = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(kh, 1),
                      stride=(kh, 1)),
            nn.AvgPool2d(kernel_size=pool_size, ceil_mode=True),
        )

        self.norm = nn.LayerNorm(out_channels)

        def forward(self, x):
            skip = self.skip_proj(x)

            y = self.conv(x)
            y = self.pool(y)
            y = self.se(y)

            # Defensive crop in case of pooling rounding mismatches.
            if y.shape[-1] != skip.shape[-1]:
                w = min(y.shape[-1], skip.shape[-1])
                y, skip = y[..., :w], skip[..., :w]
            if y.shape[-2] != skip.shape[-2]:
                h = min(y.shape[-2], skip.shape[-2])
                y, skip = y[..., :h, :], skip[..., :h, :]

            out = y + skip
            out = out.permute(0, 2, 3, 1)   # channels last for LayerNorm
            out = self.norm(out)
            out = out.permute(0, 3, 1, 2)
            return F.relu(out)

    # Adding the forward method back into the class scope correctly
    def forward(self, x):
        skip = self.skip_proj(x)

        y = self.conv(x)
        y = self.pool(y)
        y = self.se(y)

        if y.shape[-1] != skip.shape[-1]:
            w = min(y.shape[-1], skip.shape[-1])
            y, skip = y[..., :w], skip[..., :w]
        if y.shape[-2] != skip.shape[-2]:
            h = min(y.shape[-2], skip.shape[-2])
            y, skip = y[..., :h, :], skip[..., :h, :]

        out = y + skip
        out = out.permute(0, 2, 3, 1)  
        out = self.norm(out)
        out = out.permute(0, 3, 1, 2)
        return F.relu(out)


# --------------------------------------------------------------------------- #
# 3. Spatial Module: two stacked Asymmetric Conv (+SE) blocks
# --------------------------------------------------------------------------- #
class SpatialModule(nn.Module):
    """Extracts cross-channel (electrode) spatial features from one EEG
    segment. Kernel sizes follow Table I: (Cmax, 4) then (1, 4), both with
    dilation (1, 2)."""

    def __init__(self, n_channels, f1=16, f2=32, se_reduction=16):
        super().__init__()
        self.block1 = AsymmetricConvBlock(
            in_channels=1, out_channels=f1,
            kernel_size=(n_channels, 4), dilation=(1, 2),
            se_reduction=se_reduction,
        )
        self.block2 = AsymmetricConvBlock(
            in_channels=f1, out_channels=f2,
            kernel_size=(1, 4), dilation=(1, 2),
            se_reduction=se_reduction,
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        # x: (B, 1, C, T_segment)
        x = self.block1(x)          # -> (B, f1, 1, T')
        x = self.block2(x)          # -> (B, f2, 1, T'')
        return self.gap(x).flatten(1)   # -> (B, f2)


# --------------------------------------------------------------------------- #
# 4. Full Network (without TCN)
# --------------------------------------------------------------------------- #
class AsymSETNet(nn.Module):
    """End-to-end Asymmetric Squeeze-Excitation Network (TCN removed).

    Args:
        n_channels:   number of EEG electrode channels (Cmax), e.g. 22.
        n_segments:   number of temporal segments per window (N), e.g. 5.
        spatial_f1:   feature width of Spatial Module 1.
        spatial_f2:   feature width of Spatial Module 2.
        se_reduction: SE bottleneck reduction ratio (16 in the paper).
    """

    def __init__(self, n_channels=22, n_segments=5, spatial_f1=16,
                 spatial_f2=32, se_reduction=16):
        super().__init__()
        self.n_segments = n_segments

        self.spatial_module = SpatialModule(
            n_channels=n_channels, f1=spatial_f1, f2=spatial_f2,
            se_reduction=se_reduction,
        )
        
        # Classifier now directly maps the flattened spatial features of all segments
        self.classifier = nn.Sequential(
            nn.LayerNorm(spatial_f2 * n_segments),
            nn.Linear(spatial_f2 * n_segments, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        """
        x: raw EEG window, shape (B, C, T_total)
        returns: raw logits for the seizure class, shape (B,)
        """
        b, c, t_total = x.shape
        n = self.n_segments
        assert t_total % n == 0, "T_total must be divisible by n_segments"
        t_seg = t_total // n

        # Segment the recording: (B, C, T_total) -> (B*N, 1, C, T_seg)
        x = x.view(b, c, n, t_seg).permute(0, 2, 1, 3).contiguous()
        x = x.view(b * n, 1, c, t_seg)

        # Spatial feature extraction per segment -> (B*N, f2)
        feats = self.spatial_module(x)
        
        # Flatten all segment features into a single vector per batch -> (B, f2 * N)
        pooled = feats.view(b, -1)

        return self.classifier(pooled).squeeze(-1)       # (B,)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))



torch.manual_seed(0)

B, C, T_TOTAL = 8, 22, 5 * 256      # 5-second windows @ 256 Hz -> 1280 samples
N_SEGMENTS = 5

model = AsymSETNet(n_channels=C, n_segments=N_SEGMENTS)
x = torch.randn(B, C, T_TOTAL)
y = torch.randint(0, 2, (B,)).float()

logits = model(x)
loss_fn = nn.BCEWithLogitsLoss()
loss = loss_fn(logits, y)
loss.backward()  # verify gradients flow end-to-end

n_params = sum(p.numel() for p in model.parameters())
print(f"Input shape:            {tuple(x.shape)}")
print(f"Output (logits) shape:  {tuple(logits.shape)}")
print(f"BCE loss:               {loss.item():.4f}")
print(f"Trainable parameters:   {n_params:,}")

# Example optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
optimizer.step()
print("One optimizer step completed successfully.")

Input shape:            (8, 22, 1280)
Output (logits) shape:  (8,)
BCE loss:               0.6712
Trainable parameters:   4,833
One optimizer step completed successfully.


In [3]:
"""
Minimal Leave-One-Patient-Out Cross-Validation (LOPOCV) training script for
AsymSETNet on the pre-windowed CHB-MIT dataset.

Usage:
    python train_lopocv.py
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset



# --------------------------------------------------------------------------- #
# Train / evaluate a single fold
# --------------------------------------------------------------------------- #
def run_fold(X_train, y_train, X_test, y_test, n_segments=5,
             epochs=80, patience=20, batch_size=64, lr=1e-3,
             weight_decay=1e-4, device="cpu"):
    n_channels = X_train.shape[1]

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).float())
    test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).float())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = AsymSETNet(n_channels=n_channels, n_segments=n_segments).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    best_loss, best_state, no_improve = float("inf"), None, 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Validation loss (used here as the held-out fold, for early stopping)
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(loss_fn(model(xb), yb).item())
        val_loss = float(np.mean(val_losses))

        if val_loss < best_loss:
            best_loss, best_state, no_improve = val_loss, model.state_dict(), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)

    # Final evaluation: accuracy, sensitivity, specificity
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            probs = model.predict_proba(xb).cpu().numpy()
            all_preds.append((probs >= 0.5).astype(int))
            all_true.append(yb.numpy().astype(int))
    preds = np.concatenate(all_preds)
    trues = np.concatenate(all_true)

    tp = np.sum((preds == 1) & (trues == 1))
    tn = np.sum((preds == 0) & (trues == 0))
    fp = np.sum((preds == 1) & (trues == 0))
    fn = np.sum((preds == 0) & (trues == 1))

    acc = (tp + tn) / len(trues)
    sen = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return acc, sen, spec



device = "cuda" if torch.cuda.is_available() else "cpu"
X, y, patient = load_dataset()
results = {}
for pid in np.unique(patient):
    test_mask = patient == pid
    train_mask = ~test_mask
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    acc, sen, spec = run_fold(X_train, y_train, X_test, y_test, device=device, n_segments=3)
    results[pid] = (acc, sen, spec)
    print(f"Patient {pid}: Acc={acc:.4f}  Sen={sen:.4f}  Spec={spec:.4f}")
accs = [v[0] for v in results.values()]
sens = [v[1] for v in results.values()]
specs = [v[2] for v in results.values()]
print("\n=== LOPOCV summary ===")
print(f"Mean Acc:  {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print(f"Mean Sen:  {np.mean(sens):.4f} +/- {np.std(sens):.4f}")
print(f"Mean Spec: {np.mean(specs):.4f} +/- {np.std(specs):.4f}")

Patient 0: Acc=0.8630  Sen=1.0000  Spec=0.7260
Patient 1: Acc=0.8772  Sen=0.8070  Spec=0.9474
Patient 2: Acc=0.8511  Sen=0.7557  Spec=0.9466
Patient 3: Acc=0.8160  Sen=0.9440  Spec=0.6880
Patient 4: Acc=0.5541  Sen=0.9514  Spec=0.1568
Patient 5: Acc=0.6562  Sen=0.5000  Spec=0.8125
Patient 6: Acc=0.9019  Sen=0.9626  Spec=0.8411
Patient 7: Acc=0.7033  Sen=0.6230  Spec=0.7836
Patient 8: Acc=0.7278  Sen=0.9778  Spec=0.4778
Patient 9: Acc=0.9178  Sen=0.8699  Spec=0.9658
Patient 10: Acc=0.9307  Sen=0.9326  Spec=0.9288
Patient 11: Acc=0.6053  Sen=0.3270  Spec=0.8836
Patient 12: Acc=0.5106  Sen=0.1620  Spec=0.8592
Patient 13: Acc=0.5577  Sen=0.2115  Spec=0.9038
Patient 14: Acc=0.7180  Sen=0.4985  Spec=0.9375
Patient 15: Acc=0.7857  Sen=0.7619  Spec=0.8095
Patient 16: Acc=0.7990  Sen=0.6598  Spec=0.9381
Patient 17: Acc=0.8932  Sen=0.8058  Spec=0.9806
Patient 18: Acc=0.8141  Sen=0.6538  Spec=0.9744
Patient 19: Acc=0.8053  Sen=0.6632  Spec=0.9474
Patient 20: Acc=0.8462  Sen=0.7231  Spec=0.9692
Pa

# Transfer Learning

In [12]:
"""
LOPOCV training for AsymSENet with per-patient classifier fine-tuning.

Protocol per fold (target patient P):
  1. Pretrain the full model on all patients except P.
  2. Freeze everything except the classifier.
  3. Fine-tune the classifier on a held-out slice of P's NORMAL windows only
     (plus a replay buffer of seizure windows from the training patients, to
     stop the one-class objective from collapsing the decision boundary).
  4. Evaluate on P's remaining windows (unused normals + all seizures).

Assumes `asymsenet.py` exposes AsymSETNet (the TCN-free variant).

Usage:
    python train_lopocv_finetune.py
"""

import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset



SEED = 0
ADAPT_FRAC = 0.5      # fraction of the target's normal windows used for adaptation
REPLAY_PER_BATCH = 32  # seizure windows replayed from training patients
ANCHOR_LAMBDA = 1e-2   # L2 pull toward the pretrained classifier weights


# --------------------------------------------------------------------------- #
# Data
# --------------------------------------------------------------------------- #
def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y).float())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            p = model.predict_proba(xb.to(device)).cpu().numpy()
            preds.append((p >= 0.5).astype(int))
            trues.append(yb.numpy().astype(int))
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    tp = int(((preds == 1) & (trues == 1)).sum())
    tn = int(((preds == 0) & (trues == 0)).sum())
    fp = int(((preds == 1) & (trues == 0)).sum())
    fn = int(((preds == 0) & (trues == 1)).sum())

    acc = (tp + tn) / max(len(trues), 1)
    sen = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    f1 = (2 * prec * sen / (prec + sen)
          if prec == prec and sen == sen and (prec + sen) > 0 else float("nan"))
    return {"acc": acc, "sen": sen, "spec": spec, "f1": f1}


# --------------------------------------------------------------------------- #
# Stage 1: pretrain on all non-target patients
# --------------------------------------------------------------------------- #
def pretrain(X_tr, y_tr, n_segments, device,
             epochs=80, patience=20, batch_size=64,
             lr=1e-3, weight_decay=1e-4, val_frac=0.1):
    rng = np.random.default_rng(SEED)
    idx = rng.permutation(len(X_tr))
    n_val = max(int(len(idx) * val_frac), 1)
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    train_loader = make_loader(X_tr[tr_idx], y_tr[tr_idx], batch_size, shuffle=True)
    val_loader = make_loader(X_tr[val_idx], y_tr[val_idx], batch_size)

    model = AsymSETNet(n_channels=X_tr.shape[1], n_segments=n_segments).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    best, best_state, stale = float("inf"), None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl.append(loss_fn(model(xb.to(device)), yb.to(device)).item())
        vl = float(np.mean(vl))

        if vl < best:
            best, best_state, stale = vl, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    return model


# --------------------------------------------------------------------------- #
# Stage 2: classifier-only fine-tuning on the target's normal windows
# --------------------------------------------------------------------------- #
def finetune_classifier(model, X_adapt, X_replay_seiz, device,
                        epochs=10, batch_size=64, lr=1e-4):
    """X_adapt: target patient's NORMAL windows (label 0).
       X_replay_seiz: seizure windows from training patients (label 1),
       included so the one-class objective cannot collapse the boundary."""
    # Freeze the feature extractor; only the classifier head stays trainable.
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    anchor = [p.detach().clone() for p in model.classifier.parameters()]

    y_adapt = np.zeros(len(X_adapt), dtype=np.float32)
    adapt_loader = make_loader(X_adapt, y_adapt, batch_size, shuffle=True)

    replay_X = torch.from_numpy(X_replay_seiz)
    rng = np.random.default_rng(SEED)

    opt = torch.optim.Adam(model.classifier.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    # Keep the frozen trunk in eval mode (no running-stat / dropout drift).
    model.spatial_module.eval()

    for _ in range(epochs):
        for xb, yb in adapt_loader:
            xb, yb = xb.to(device), yb.to(device)

            if len(replay_X) > 0:
                k = min(REPLAY_PER_BATCH, len(replay_X))
                sel = rng.choice(len(replay_X), size=k, replace=False)
                rx = replay_X[sel].to(device)
                ry = torch.ones(k, device=device)
                xb = torch.cat([xb, rx], dim=0)
                yb = torch.cat([yb, ry], dim=0)

            opt.zero_grad()
            loss = loss_fn(model(xb), yb)

            # Anchor: penalise drift away from the pretrained head.
            reg = sum(((p - a) ** 2).sum()
                      for p, a in zip(model.classifier.parameters(), anchor))
            (loss + ANCHOR_LAMBDA * reg).backward()
            delta = sum((p - a).abs().sum().item()
            for p, a in zip(model.classifier.parameters(), anchor))

            opt.step()

    return model


# --------------------------------------------------------------------------- #
# LOPOCV driver
# --------------------------------------------------------------------------- #
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    X, y, patient = load_dataset()
    n_segments = 3
    rng = np.random.default_rng(SEED)

    rows = []
    for pid in np.unique(patient):
        test_mask = patient == pid
        X_tr, y_tr = X[~test_mask], y[~test_mask]
        X_te, y_te = X[test_mask], y[test_mask]

        # --- split the target's NORMAL windows into adapt / test halves ---
        normal_idx = np.where(y_te == 0)[0]
        seizure_idx = np.where(y_te == 1)[0]
        normal_idx = rng.permutation(normal_idx)
        n_adapt = int(len(normal_idx) * ADAPT_FRAC)
        adapt_idx = normal_idx[:n_adapt]
        held_normal_idx = normal_idx[n_adapt:]

        eval_idx = np.concatenate([held_normal_idx, seizure_idx])
        X_eval, y_eval = X_te[eval_idx], y_te[eval_idx]
        X_adapt = X_te[adapt_idx]

       
        
        eval_loader = make_loader(X_eval, y_eval)



        # --- stage 1: pretrain ---
        model = pretrain(X_tr, y_tr, n_segments, device)
        before = evaluate(model, eval_loader, device)

        # --- stage 2: classifier-only adaptation ---
        replay_seiz = X_tr[y_tr == 1]
        if len(replay_seiz) > 512:
            replay_seiz = replay_seiz[rng.choice(len(replay_seiz), 512, replace=False)]
        model = finetune_classifier(model, X_adapt, replay_seiz, device)

                # before and after finetune_classifier
        with torch.no_grad():
            p = torch.cat([model.predict_proba(xb.to(device)).cpu()
            for xb, _ in eval_loader]).numpy()
            print(f"   mean|p-0.5|={np.abs(p-0.5).mean():.3f}  "
            f"frac within 0.05 of threshold={np.mean(np.abs(p-0.5)<0.05):.3f}")
        after = evaluate(model, eval_loader, device)

        rows.append((pid, before, after))
        print(f"Patient {pid} | n_adapt={len(X_adapt)} n_eval={len(y_eval)}")
        print(f"   before  acc={before['acc']:.4f} sen={before['sen']:.4f} "
              f"spec={before['spec']:.4f} f1={before['f1']:.4f}")
        print(f"   after   acc={after['acc']:.4f} sen={after['sen']:.4f} "
              f"spec={after['spec']:.4f} f1={after['f1']:.4f}")

    # --- summary ---
    def agg(stage, key):
        vals = [r[stage][key] for r in rows]
        vals = [v for v in vals if v == v]  # drop NaNs
        return np.mean(vals), np.std(vals)

    print("\n=== LOPOCV summary (mean +/- std across patients) ===")
    print(f"{'metric':<6} {'before':>18} {'after':>18}")
    for key in ("acc", "sen", "spec", "f1"):
        bm, bs = agg(1, key)
        am, asd = agg(2, key)
        print(f"{key:<6} {bm:>10.4f} +/- {bs:.4f} {am:>10.4f} +/- {asd:.4f}")


main()

   mean|p-0.5|=0.414  frac within 0.05 of threshold=0.032
Patient 0 | n_adapt=73 n_eval=219
   before  acc=0.9132 sen=0.9521 spec=0.8356 f1=0.9360
   after   acc=0.9132 sen=0.9521 spec=0.8356 f1=0.9360
   mean|p-0.5|=0.429  frac within 0.05 of threshold=0.023
Patient 1 | n_adapt=28 n_eval=86
   before  acc=0.9302 sen=0.9123 spec=0.9655 f1=0.9455
   after   acc=0.9186 sen=0.9123 spec=0.9310 f1=0.9369
   mean|p-0.5|=0.409  frac within 0.05 of threshold=0.025
Patient 2 | n_adapt=65 n_eval=197
   before  acc=0.8883 sen=0.8321 spec=1.0000 f1=0.9083
   after   acc=0.8883 sen=0.8321 spec=1.0000 f1=0.9083
   mean|p-0.5|=0.463  frac within 0.05 of threshold=0.005
Patient 3 | n_adapt=62 n_eval=188
   before  acc=0.8723 sen=0.9440 spec=0.7302 f1=0.9077
   after   acc=0.8723 sen=0.9440 spec=0.7302 f1=0.9077
   mean|p-0.5|=0.411  frac within 0.05 of threshold=0.040
Patient 4 | n_adapt=92 n_eval=278
   before  acc=0.7266 sen=0.9838 spec=0.2151 f1=0.8273
   after   acc=0.7266 sen=0.9784 spec=0.2258 f